In [25]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [26]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [27]:
df.shape

(5000, 9)

In [28]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [29]:
for col in df.columns:
    df[col] = df[col].fillna('NA') if df[col].dtype == 'object' else df[col].fillna(0.0)
df.isnull().sum().sum()

np.int64(0)

In [30]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
0.0               240
Name: count, dtype: int64

In [31]:
df['industry'].mode()[0]

'technology'

In [32]:
numcols = df.select_dtypes(include=[np.number]).columns.tolist()
numcols

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [33]:
corrmat = df[numcols].corr()
corrmat

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [34]:
dffull, dftest = train_test_split(df, test_size=0.2, random_state=42)
dftr, dfvl = train_test_split(dffull, test_size=0.25, random_state=42)

dftr = dftr.reset_index(drop=True)
dfvl = dfvl.reset_index(drop=True)
dftest = dftest.reset_index(drop=True)

ytr = dftr['converted'].values
yvl = dfvl['converted'].values
yts = dftest['converted'].values

del dftr['converted'], dfvl['converted'], dftest['converted']

dftr.shape, dfvl.shape, dftest.shape

((3000, 8), (1000, 8), (1000, 8))

In [35]:
catcols = dftr.select_dtypes(include='object').columns.tolist()
catcols

['lead_source', 'industry', 'employment_status', 'location']

In [36]:
ytr = ytr.astype(int)
{c: round(mutual_info_score(ytr, dftr[c].astype(str)), 2) for c in catcols}

{'lead_source': 0.03,
 'industry': 0.0,
 'employment_status': 0.02,
 'location': 0.0}

In [37]:
dv = DictVectorizer(sparse=False)
Xtr = dv.fit_transform(dftr.to_dict(orient='records'))
Xvl = dv.transform(dfvl.to_dict(orient='records'))

In [38]:
mdl = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
mdl.fit(Xtr, ytr)
acc = accuracy_score(yvl, mdl.predict(Xvl))
round(acc, 2)

0.65

In [39]:
base = accuracy_score(yvl, mdl.predict(Xvl))
base

0.645

In [40]:
res = {}
for f in dftr.columns:
    trsub = dftr.drop(columns=[f])
    vlsub = dfvl.drop(columns=[f])

    dv2 = DictVectorizer(sparse=False)
    Xtr2 = dv2.fit_transform(trsub.to_dict(orient='records'))
    Xvl2 = dv2.transform(vlsub.to_dict(orient='records'))

    mdl2 = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    mdl2.fit(Xtr2, ytr)

    res[f] = base - accuracy_score(yvl, mdl2.predict(Xvl2))

res

{'lead_source': 0.0030000000000000027,
 'industry': 0.0,
 'employment_status': 0.0,
 'location': 0.0,
 'annual_income': -0.07999999999999996,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004,
 'lead_score': 0.0010000000000000009}

In [41]:
{f: res[f] for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count'] if f in res}

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

In [42]:
resC = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    mc = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    mc.fit(Xtr, ytr)
    resC[C] = round(accuracy_score(yvl, mc.predict(Xvl)), 3)
resC

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}